# Jupyter notebook on obtaining a whole-brain inter-voxel cross-correlation matrix
*By Joshua Goh, 22 Nov 2023*

## Overview
Most brain functional connectivity studies examine brain inter-regional functional MR BOLD signal time series correlations. This is typically applied by first using regions-of-interest (ROI) mask definitions, such as from the AAL template, or even functionally defined ROIs. However, ROIs suffer from the problem that a hard boundary (even if defined from probabilistic maps) is eventually imposed on a given participant's brain time instance, and the average time series from these partitioned brain voxel grouping used for functional connectivity. This may not be accurate partitioning of the brain's true functional modules (if they exist in that moment in time). Moreover, average ROI signals are in principle non-existent -- they reflect the equal contribution of each voxel, but do not reflect the inherent dynamic variability in voxel responses. In keeping with the Brain World project, we thus require an approach that avoids such false constraints imposed on the data. In this Jupyter Notebook, we work out a method to compute the whole-brain inter-voxel cross-correlation matrix without the use of ROIs.

## Data size considerations
Note that this Jupyter Notebook works for cross-correlation matrices that are within the size of the system RAM. For whole-brain inter-voxel cross-correlation matrices that exceed system RAM, it might be more suitable use the approach found in [corr_matrix_clustering.ipynb]().

# Setup environment

## Install dependencies
Below are the package dependencies required for the functions used in this notebook. This only needs to be installed once in local systems or environments. For cloud systems (e.g. Google Colab), installation is required each refreshed runtime. Run as needed for your environment.

In [1]:
#! pip install numpy
#! pip install -U nilearn
#! pip uninstall niwidgets
#! pip install --upgrade --force-reinstall nibabel
#! pip install pandas
#! pip install nibabel
#! pip install seaborn
#! pip install -I plotly
#!pip install ipywidgets
#! pip install scipy
#! pip install scikit-image
#! pip install matplotlib
#!pip install tifffile

## Import package functions
If the dependencies are all installed, below are the function imports required for this Notebook runtime.

In [2]:
import numpy as np
import nibabel as nib
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import tifffile
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
from IPython.display import display

## Read and format brain data
The nibabel package functions are used to read the input brain image volume. Example path and file, 'Filtered_4DVolume.nii', are used here, which refer to a preprocessed resting state functional 4D volume. You should change the file path to suit your data accordingly. Note that the 4D brain image is read into a 4D array, a variance volume is obtained, the main data is then reshaped to be a voxel x time point 2D array.

In [3]:
brain_img_file = '/home/joshgoh/Projects/brain-world/data/derivatives/5129/mri/func/rest/1st-level-results/Res_detrend_filtered/Filtered_4DVolume.nii'
brain_vol = nib.load(brain_img_file)
Y = brain_vol.get_fdata()
S = np.var(Y,axis=3) # Variance volume

# Reshape brain image into array form
Y = np.reshape(Y,(np.prod(Y.shape[0:3]),Y.shape[3]))

## Read brain image mask
The full brain volume array contains many non-brain tissue voxels. These are masked out to improve efficiency. Masking is first applied externally on the resting state 4D volume using SPM to include gray matter (GM) segmented C1*.nii file (from SPM's segment process). This masked data volume is then compressed to a 3D volume (considering all voxels with possible signal throughout the time series), and then binarized (1 GM, 0 non-GM). Here, we further add the consideration that within this binarized GM mask, we also only consider voxels where the time series variances are not zero.

In [4]:
# Define masked search voxel set
brain_mask_file = '/home/joshgoh/Projects/brain-world/data/derivatives/5129/mri/func/rest/1st-level-results/Res_detrend_filtered/bmFiltered_4DVolume.nii'
brain_mask_vol = nib.load(brain_mask_file)
M = brain_mask_vol.get_fdata()

I = np.reshape(np.arange(0,np.prod(M.shape)),(M.shape[0],M.shape[1],M.shape[2])) # GM mask flat indices

vset = I[np.where((M!=0) & (S!=0))]

# Apply GM mask indices on data
MY = Y[vset,:]

## Write new brain image mask
This new binary mask updates the original one to further only include voxels where the functional time series variance is > 0 (i.e., only where there are MR signals within the GM mask originally applied). This is critical for the correlation operation since, as per the correlation formula, it  will involve dividing covariances by the data variances. If data variance (the divisor) is zero, the correlation function will throw and error.

In [5]:
# For registration for Dijkstra distance
bnM = np.zeros(M.shape) # New binary mask array
bnM[np.where((M!=0) & (S!=0))] = 1

bnM_vol = nib.Nifti1Image(bnM,brain_mask_vol.affine,brain_mask_vol.header)
nib.save(bnM_vol,'/home/joshgoh/Projects/brain-world/data/derivatives/5129/mri/func/rest/1st-level-results/Res_detrend_filtered/bnbmFiltered_4DVolume.nii')

## Visualize mask and data

In [ ]:
# Plot mask
BV = bnM.copy() # Assign brain volume to show

x,y,z = int(BV.shape[1]/2), int(BV.shape[0]/2), int(BV.shape[2]/2) # Start at midpoint coordinates

fig = go.FigureWidget(make_subplots(rows=1,cols=3))
fig.update_layout(coloraxis=dict(colorscale='gray'),margin=dict(l=0, r=5, t=5, b=5),height=200, width=800,showlegend=False,plot_bgcolor='rgba(255,255,255,1)',paper_bgcolor='rgba(255,255,255,1)')
fig.add_trace(go.Heatmap(z=BV[:,x,:].T,coloraxis='coloraxis'),row=1, col=1)
fig.add_trace(go.Heatmap(z=BV[y,:,:].T,coloraxis='coloraxis'),row=1, col=2)
fig.add_trace(go.Heatmap(z=BV[:,:,z].T,coloraxis='coloraxis'),row=1, col=3)

cor,sag,tra = fig.data[0], fig.data[1], fig.data[2]

# Coordinate definition functions
def xc(x):
    cor.z=BV[:,x,:].T
def yc(y):
    sag.z=BV[y,:,:].T
def zc(z):
    tra.z=BV[:,:,z].T

xw,yw,zw = interactive(xc, x=(0,BV.shape[1]-1)),interactive(yc, y=(0,BV.shape[0]-1)),interactive(zc, z=(0,BV.shape[2]-1))

print("Orthographic view of mask (GM, no non-zero time-series)")
display(xw),display(yw),display(zw)
fig

Below shows a sample figure of the output if the above cell is run.
![Brain mask volume image](brain_vx_corr_matrix_media/Brain_Mask.png)

In [ ]:
# Plot masked data
print("Raster plot of voxel (rows) time-series (columns) data, the reshaped Y, extracted from the above mask (GM, no non-zero time-series)")
fig = go.Figure(data=go.Heatmap(z=MY,colorscale='RdBu_r'))
fig.update_layout(margin=dict(l=0, r=5, t=5, b=5),height=200, width=800,showlegend=False,plot_bgcolor='rgba(255,255,255,1)',paper_bgcolor='rgba(255,255,255,1)')
fig.update_yaxes(showticklabels=True,showgrid=False)
fig.show(config={'displayModeBar':False})

Below shows a sample figure of the output if the above cell is run.
![Data_Raster_Plot](brain_vx_corr_matrix_media/Data_Raster_Plot.png)

# Begin

## Calculate inter-voxel cross-correlation matrix
This is the core computation applied on the 2D Y array (v, t).

In [10]:
R = np.corrcoef(MY)
np.savez_compressed('/home/joshgoh/Projects/brain-world/data/derivatives/5129/mri/func/rest/fc-results/R.npz',R)

## Visualize correlation matrix

In [ ]:
# Plot partial range, 500*500 recommended size.
i,j = 0,500 # Initial start voxel, range 

print("Partial plot of inter-voxel correlation matrix: Start voxel ",str(i),", range ",str(j))
fig = go.Figure(data=go.Heatmap(z=R[i:i+j,i:i+j],colorscale='RdBu_r'))
fig.update_layout(margin=dict(l=0, r=5, t=5, b=5),height=500,width=600,showlegend=False,plot_bgcolor='rgba(255,255,255,1)',paper_bgcolor='rgba(255,255,255,1)')
fig.update_yaxes(showticklabels=True,showgrid=False)
fig.show()

Below shows a sample figure of the output if the above cell is run.
![Correlation matrix](brain_vx_corr_matrix_media/Correlation_Matrix.png)

In [12]:
# Write full matrix to tiff file.
tifffile.imwrite('/home/joshgoh/Projects/brain-world/data/derivatives/5129/mri/func/rest/fc-results/R_rs.tif',
                 np.uint8(255/2*(R+1)),
                 bigtiff=True,
                 photometric='minisblack',
                 compression='zlib')

# Conclusion
The R.npz file containing the whole-brain inter-voxel cross-correlations can be used in subsequent further analyses. E.g. cluster analysis, or association with inter-voxel physical distance (see [Dijkstra distance ipynb](https://gitlab.com/joshuagoh/brain-world/-/blob/main/code/dijkstra_brain/dijkstra_brain.ipynb))